# 효율적 투자선 — 모의 트레이딩 하네스 투자 가능 자산 전체 + 원화 현금 (롱·숏)

- **대상**: `투자가능자산.md`의 현물 자산(코인 20 / 나스닥 105 / 코스피 104) + **원화 현금**. 선물은 제외
  (숏은 코인·나스닥 현물로 이미 가능하고, 무기한 선물 6종은 기초자산이 모두 현물 코인(BTC·ETH·SOL·XRP)과 겹치며, 펀딩비·청산 위험이 있어 평균-분산 틀에 맞지 않음)
- **숏(공매도)**: 코인·나스닥 현물은 가능, 코스피는 불가. 숏 매도대금은 해당 통화(USD·USDT)로 남으므로
  숏 포지션의 원화 수익률 ≈ −(현지통화 수익률). 즉 숏은 환율 노출이 상쇄됨
- **현금**: 하네스의 원화 현금은 이자가 없으므로 수익률 0%, 변동성 0%, 비용 0인 자산으로 포함. 무위험수익률도 0%
- **데이터**: yfinance (외부 데이터). 분배금을 주지 않는 하네스 규칙에 맞춰 배당 미조정 종가(Close) 사용
- **기준통화 KRW**: 달러·USDT 자산 롱은 USD/KRW 환율을 곱해 원화 수익률로 계산 (USDT ≈ USD 로 가정)
- **주간 수익률**: 장 시간이 다른 시장을 섞기 위해 금요일 종가 기준
- **비용 반영**: 롱은 원화 → 매수 → 매도 → 원화 왕복 비용, 숏은 매도 → 환매수 매매 비용(환전 없음 가정) + 대여비용 `SHORT_BORROW`를
  보유기간으로 나눠 기대수익률에서 차감. 숏 대여비용·증거금 규칙은 문서에 없음 → `get_rules`로 확인 필요
- **제약**: 롱 비중 + 현금 = 1, 롱·숏 절대값 합 ≤ 총자산 × 1배 (하네스 노출 한도, 차입 불가),
  종목당 롱 ≤ `MAX_WEIGHT`, 숏 ≤ `MAX_SHORT` (`MAX_SHORT = 0`이면 롱온리)

필요 패키지: `pip install yfinance matplotlib scipy pandas numpy`

## 1. 옵션

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.optimize import linprog, minimize

LOOKBACK_YEARS = 3          # 과거 몇 년 데이터로 추정할지
MIN_COVERAGE = 0.95         # 기간 중 이 비율 이상 가격이 있는 자산만 사용 (신규 상장 종목 제외)
HOLDING_YEARS = 1.0         # 가정 보유기간. 왕복 비용 / 보유기간 = 연간 비용
RF_ANNUAL = 0.0             # 무위험수익률 = 현금 수익률 (하네스 현금은 이자 없음)
MAX_WEIGHT = 1.0            # 위험자산 한 종목 최대 롱 비중 (예: 0.2 → 20% 상한). 현금에는 적용 안 함
MAX_SHORT = 0.10            # 한 종목 최대 숏 비중 (0이면 롱온리). 코인·나스닥만, 코스피는 공매도 불가
SHORT_BORROW = 0.0          # 숏 연간 대여비용 (규칙에 명시 없음 → get_rules로 확인)
GROSS_LIMIT = 1.0           # 롱·숏 절대값 합 ≤ 총자산 × 배수 (하네스 현물 레버리지 1배)
N_FRONTIER = 40             # 효율적 투자선 곡선 구간(접점 위)에 찍을 점 개수
COV_SHRINK = True           # Ledoit-Wolf 공분산 축소 (자산 수 > 관측치 수라 표본공분산은 불안정)
APPLY_COST = True           # 거래·환전 비용 반영 여부
LABEL_TOP = 12              # 그래프에 이름을 표시할 상위 비중 포지션 수
CASH = "KRW"                # 현금 자산 이름

OUT_DIR = Path.cwd()
OUT_PNG = OUT_DIR / "efficient_frontier.png"
OUT_CSV = OUT_DIR / "efficient_frontier_weights.csv"

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

## 2. 거래 비용 (get_rules 기준)
편도 = 수수료 + 슬리피지, 환전은 원화 → 해당 시장 통화.
- 롱 왕복 = 2 × (매매 편도 + 환전 편도)
- 숏 왕복 = 2 × 매매 편도 (매도대금이 해당 통화로 들어와 환매수에 그대로 쓰이므로 환전 없음 가정)

In [2]:
TRADE_ONE_WAY = {"kospi": 0.0015 + 0.0005, "nasdaq": 0.0005 + 0.0003, "crypto": 0.0010 + 0.0002}
FX_ONE_WAY = {"kospi": 0.0, "nasdaq": 0.0090, "crypto": 0.0090 + 0.0015}  # KRW→USD, KRW→USD→USDT
ROUND_TRIP = {m: 2 * (TRADE_ONE_WAY[m] + FX_ONE_WAY[m]) for m in TRADE_ONE_WAY}
ROUND_TRIP["cash"] = 0.0
SHORTABLE = {"nasdaq", "crypto"}                                           # 코스피는 공매도 불가
SHORT_ROUND_TRIP = {m: 2 * TRADE_ONE_WAY[m] for m in SHORTABLE}
pd.DataFrame({"롱 왕복": pd.Series(ROUND_TRIP), "숏 왕복": pd.Series(SHORT_ROUND_TRIP)}).map(
    lambda x: "불가" if pd.isna(x) else f"{x:.2%}")

,롱 왕복,숏 왕복
cash,0.00%,불가
crypto,2.34%,0.24%
kospi,0.40%,불가
nasdaq,1.96%,0.16%


## 3. 투자 가능 자산 (하네스 심볼 → yfinance 티커)

In [3]:
CRYPTO = {
    "ADAUSDT": "ADA-USD", "AVAXUSDT": "AVAX-USD", "BCHUSDT": "BCH-USD", "BNBUSDT": "BNB-USD",
    "BTCUSDT": "BTC-USD", "DOGEUSDT": "DOGE-USD", "DOTUSDT": "DOT-USD", "ETHUSDT": "ETH-USD",
    "HBARUSDT": "HBAR-USD", "HYPEUSDT": "HYPE32196-USD", "LINKUSDT": "LINK-USD", "LTCUSDT": "LTC-USD",
    "NEARUSDT": "NEAR-USD", "SOLUSDT": "SOL-USD", "SUIUSDT": "SUI20947-USD", "TONUSDT": "TON11419-USD",
    "TRXUSDT": "TRX-USD", "UNIUSDT": "UNI7083-USD", "XLMUSDT": "XLM-USD", "XRPUSDT": "XRP-USD",
}

NASDAQ = """
AAPL ABNB ADBE ADI ADP ADSK AEP ALAB ALNY AMAT AMD AMGN AMZN APP ARM ASML AVGO AXON BKNG BKR
CCEP CDNS CEG CMCSA COIN COST CRWD CRWV CSCO CSX CTAS DASH DDOG DXCM EXC FANG FAST FER FTNT GEHC
GILD GOOG GOOGL HON HONA IDXX IEF INTC INTU ISRG KDP KLAC LIN LITE LRCX MAR MCHP MDLZ MELI META
MNST MPWR MRNA MRVL MSFT MSTR MU NBIS NFLX NVDA NXPI ODFL ORLY PANW PAYX PCAR PDD PEP PLTR PYPL
QCOM REGN RKLB ROP ROST SBUX SHOP SHY SMCI SNDK SNPS SPCX STX TER TLT TMUS TRI TSLA TTWO TXN
VRTX WDAY WDC WMT XEL
""".split()

KOSPI = {
    "000100": "유한양행", "000150": "두산", "000270": "기아", "000500": "가온전선", "000660": "SK하이닉스",
    "000720": "현대건설", "000810": "삼성화재", "000880": "한화", "001440": "대한전선", "003230": "삼양식품",
    "003490": "대한항공", "003550": "LG", "003670": "포스코퓨처엠", "005380": "현대차", "005490": "POSCO홀딩스",
    "005830": "DB손해보험", "005930": "삼성전자", "005940": "NH투자증권", "006260": "LS", "006400": "삼성SDI",
    "006800": "미래에셋증권", "007660": "이수페타시스", "009150": "삼성전기", "009540": "HD한국조선해양",
    "009830": "한화솔루션", "010120": "LS ELECTRIC", "010130": "고려아연", "010140": "삼성중공업",
    "010950": "S-Oil", "011070": "LG이노텍", "011200": "HMM", "011790": "SKC", "012330": "현대모비스",
    "012450": "한화에어로스페이스", "015760": "한국전력", "016360": "삼성증권", "017670": "SK텔레콤",
    "018260": "삼성에스디에스", "021240": "코웨이", "024110": "기업은행", "028050": "삼성E&A",
    "028260": "삼성물산", "029780": "삼성카드", "030200": "KT", "032640": "LG유플러스", "032830": "삼성생명",
    "033780": "KT&G", "034020": "두산에너빌리티", "034730": "SK", "035420": "NAVER", "035720": "카카오",
    "036570": "엔씨소프트", "039490": "키움증권", "042660": "한화오션", "042700": "한미반도체",
    "047040": "대우건설", "047050": "포스코인터내셔널", "047810": "한국항공우주", "051910": "LG화학",
    "055550": "신한지주", "062040": "산일전기", "064350": "현대로템", "064400": "LG씨엔에스",
    "066570": "LG전자", "068270": "셀트리온", "071050": "한국금융지주", "078930": "GS", "079550": "LIG넥스원",
    "086280": "현대글로비스", "086790": "하나금융지주", "088980": "맥쿼리인프라", "090430": "아모레퍼시픽",
    "096770": "SK이노베이션", "105560": "KB금융", "114260": "KODEX 국고채3년", "128940": "한미약품",
    "138040": "메리츠금융지주", "138930": "BNK금융지주", "161390": "한국타이어앤테크놀로지",
    "175330": "JB금융지주", "180640": "한진칼", "207940": "삼성바이오로직스", "241560": "두산밥캣",
    "251270": "넷마블", "259960": "크래프톤", "267250": "HD현대", "267260": "HD현대일렉트릭",
    "271560": "오리온", "272210": "한화시스템", "278470": "에이피알", "298040": "효성중공업",
    "307950": "현대오토에버", "316140": "우리금융지주", "323410": "카카오뱅크", "326030": "SK바이오팜",
    "329180": "HD현대중공업", "352820": "하이브", "353200": "대덕전자", "365780": "ACE 국고채10년",
    "373220": "LG에너지솔루션", "377300": "카카오페이", "385560": "RISE 국고채30년", "402340": "SK스퀘어",
    "443060": "HD현대마린솔루션",
}

MARKET_COLOR = {"kospi": "#1f77b4", "nasdaq": "#2ca02c", "crypto": "#ff7f0e"}
MARKET_LABEL = {"kospi": "코스피", "nasdaq": "나스닥", "crypto": "코인", "cash": "현금"}

rows = [(s, t, "crypto", s) for s, t in CRYPTO.items()]
rows += [(s, s, "nasdaq", s) for s in NASDAQ]
rows += [(c, f"{c}.KS", "kospi", n) for c, n in KOSPI.items()]
universe = pd.DataFrame(rows, columns=["symbol", "yf", "market", "name"]).set_index("symbol")
universe.loc[CASH] = [None, "cash", "원화 현금"]
universe["market"].value_counts()

market
nasdaq    105
kospi     104
crypto     20
cash        1
Name: count, dtype: int64

## 4. 가격 내려받기 (yfinance, 배당 미조정 종가)

In [ ]:
import shutil
import tempfile
import warnings

import certifi
from curl_cffi import requests as curl_requests

# curl_cffi는 인증서 경로에 한글이 있으면 읽지 못함(curl 77 SSLError) → 영문 경로(임시 폴더)로 복사해서 사용
CA_FILE = Path(tempfile.gettempdir()) / "yf_cacert.pem"
shutil.copyfile(certifi.where(), CA_FILE)
yf_session = curl_requests.Session(impersonate="chrome", verify=str(CA_FILE))
warnings.filterwarnings("ignore", module="yfinance")   # yfinance 내부의 pandas 사용 중단 경고

risky = universe[universe["market"] != "cash"]
end = pd.Timestamp.today().normalize()
start = end - pd.DateOffset(years=LOOKBACK_YEARS)
raw = yf.download(risky["yf"].tolist() + ["USDKRW=X"], start=start, end=end + pd.Timedelta(days=1),
                  auto_adjust=False, progress=False, threads=True, session=yf_session)["Close"]
fx = raw.pop("USDKRW=X")
px = raw.rename(columns={v: k for k, v in risky["yf"].items()})
print(f"가격 {px.shape[1]}종, {px.index[0]:%Y-%m-%d} ~ {px.index[-1]:%Y-%m-%d}")

## 5. 주간 수익률
시장마다 거래일이 달라서 금요일 기준 주간 종가로 맞춤.
- `base`: 현지통화 수익률 + USD/KRW 수익률 (공분산 추정의 기초)
- `rets`: 원화 기준 수익률 = (1 + 현지통화 수익률)(1 + 환율 수익률) − 1 (롱 포지션의 기대수익률)

In [ ]:
wpx = px.resample("W-FRI").last()
wfx = fx.resample("W-FRI").last().ffill()

coverage = wpx.notna().mean()
keep = coverage[coverage >= MIN_COVERAGE].index
dropped = sorted(set(risky.index) - set(keep))
print(f"데이터 부족으로 제외 ({len(dropped)}개): {', '.join(dropped)}")

local = wpx[keep].ffill(limit=2).pct_change(fill_method=None)
base = pd.concat([local, wfx.pct_change().rename("USDKRW")], axis=1).iloc[1:].dropna()
usd_keep = risky.index[risky["market"] != "kospi"].intersection(keep)
rets = base[keep].copy()
rets[usd_keep] = (1 + base[usd_keep]).mul(1 + base["USDKRW"], axis=0) - 1
print(f"사용 위험자산 {rets.shape[1]}개, 주간 관측치 {rets.shape[0]}개 "
      f"({rets.index[0]:%Y-%m-%d} ~ {rets.index[-1]:%Y-%m-%d})")

## 6. 기대수익률·공분산 추정
- 포지션: 위험자산 롱 + 코인·나스닥 숏 + 원화 현금
- 공분산: 기초 수익률(`base`)에 Ledoit & Wolf (2004) 상수상관 타깃 축소를 한 뒤, 포지션 = 기초 수익률의 선형결합으로 변환
  - 롱(달러 자산) = 현지통화 수익률 + 환율 수익률, 롱(코스피) = 현지통화 수익률, 숏 = −현지통화 수익률, 현금 = 0
  - 같은 종목의 롱·숏을 별개 자산으로 놓고 축소하면 상관이 −1에서 멀어져 가짜 분산효과가 생기므로 이렇게 처리
- 기대수익률: 롱 = 원화 수익률 평균 × 52 − 연간 비용, 숏 = −현지통화 수익률 평균 × 52 − 연간 비용 − 대여비용

In [ ]:
def ledoit_wolf(x: np.ndarray) -> np.ndarray:
    t, n = x.shape
    y = x - x.mean(axis=0)
    s = y.T @ y / t
    var = np.diag(s)
    sd = np.sqrt(var)
    r_bar = (np.sum(s / np.outer(sd, sd)) - n) / (n * (n - 1))
    target = r_bar * np.outer(sd, sd)
    np.fill_diagonal(target, var)

    pi_mat = (y ** 2).T @ (y ** 2) / t - s ** 2
    pi_hat = pi_mat.sum()
    gamma_hat = np.sum((s - target) ** 2)
    theta = (y ** 3).T @ y / t - var[:, None] * s
    np.fill_diagonal(theta, 0)
    rho_hat = np.trace(pi_mat) + r_bar * np.sum(np.outer(1 / sd, sd) * theta)
    delta = max(0.0, min(1.0, (pi_hat - rho_hat) / gamma_hat / t))
    print(f"Ledoit-Wolf 축소 강도: {delta:.3f} (평균 상관 {r_bar:.3f})")
    return delta * target + (1 - delta) * s


cov_base = ledoit_wolf(base.values) * 52 if COV_SHRINK else base.cov().values * 52

mkt_r = universe.loc[rets.columns, "market"]
shorts = rets.columns[mkt_r.isin(SHORTABLE)] if MAX_SHORT > 0 else rets.columns[:0]
pos = pd.DataFrame({
    "symbol": list(rets.columns) + list(shorts) + [CASH],
    "side": ["롱"] * len(rets.columns) + ["숏"] * len(shorts) + ["현금"],
})
pos.index = [f"{s} 숏" if d == "숏" else s for s, d in zip(pos["symbol"], pos["side"])]
pos["market"] = universe.loc[pos["symbol"], "market"].values
pos["name"] = universe.loc[pos["symbol"], "name"].values
side = pos["side"].values
is_long, is_short, is_cash = side == "롱", side == "숏", side == "현금"

col = {c: i for i, c in enumerate(base.columns)}
A = np.zeros((len(pos), len(col)))
for k, (s, d, m) in enumerate(zip(pos["symbol"], side, pos["market"])):
    if d == "롱":
        A[k, col[s]] = 1
        if m != "kospi":
            A[k, col["USDKRW"]] = 1
    elif d == "숏":
        A[k, col[s]] = -1
cov = A @ cov_base @ A.T

mu = np.full(len(pos), RF_ANNUAL)
mu[is_long] = rets.mean().values * 52
mu[is_short] = -base[shorts].mean().values * 52 - SHORT_BORROW
if APPLY_COST:
    mu[is_long] -= pos["market"][is_long].map(ROUND_TRIP).values / HOLDING_YEARS
    mu[is_short] -= pos["market"][is_short].map(SHORT_ROUND_TRIP).values / HOLDING_YEARS

names = pos.index
mkt = pos["market"]
vols = np.sqrt(np.diag(cov))
print(f"포지션 {len(pos)}개 = 롱 {is_long.sum()} + 숏 {is_short.sum()} + 현금 1")

## 7. 최적화
- 제약: 롱 합 + 현금 = 1, 롱 합 + 숏 합 ≤ `GROSS_LIMIT`, 0 ≤ 롱 ≤ `MAX_WEIGHT`, 0 ≤ 숏 ≤ `max_short`
- `solve(target)`: 목표 기대수익률에서 분산 최소화
- 최대샤프(접점) 포트폴리오: 노출 한도를 꽉 채운(롱 + 숏 = 1) 조합 중 샤프비율 최대. 숏 매도대금만큼 현금이 남음
- 효율적 투자선 = 원점(현금 100%) ~ 접점까지 직선 + 접점 이후 곡선. 노출 한도 때문에 접점 위로는 직선이 연장되지 않음
- 비교용으로 `max_short=0`(롱온리) 투자선도 같이 계산

In [ ]:
budget = (is_long | is_cash).astype(float)   # 롱 + 현금 = 1
gross = (is_long | is_short).astype(float)   # 롱 + 숏 ≤ GROSS_LIMIT


def bounds_for(max_short):
    ub = np.where(is_long, MAX_WEIGHT, np.where(is_short, max_short, 1.0))
    return list(zip(np.zeros(len(mu)), ub))


def max_return(max_short):
    res = linprog(-mu, A_ub=[gross], b_ub=[GROSS_LIMIT], A_eq=[budget], b_eq=[1.0],
                  bounds=bounds_for(max_short), method="highs")
    return -res.fun


def solve(target=None, max_sharpe=False, max_short=MAX_SHORT):
    cons = [{"type": "eq", "fun": lambda w: w @ budget - 1, "jac": lambda w: budget}]
    w0 = is_long / is_long.sum()
    if max_sharpe:
        cons.append({"type": "eq", "fun": lambda w: w @ gross - GROSS_LIMIT, "jac": lambda w: gross})

        def obj(w):
            v = np.sqrt(w @ cov @ w)
            return -(w @ mu - RF_ANNUAL) / v

        def jac(w):
            cw = cov @ w
            v = np.sqrt(w @ cw)
            return -(mu / v - (w @ mu - RF_ANNUAL) * cw / v ** 3)
    else:
        cons.append({"type": "ineq", "fun": lambda w: GROSS_LIMIT - w @ gross, "jac": lambda w: -gross})
        if target is not None:
            cons.append({"type": "eq", "fun": lambda w: w @ mu - target, "jac": lambda w: mu})
        obj = lambda w: w @ cov @ w
        jac = lambda w: 2 * cov @ w
    res = minimize(obj, w0, jac=jac, bounds=bounds_for(max_short), constraints=cons, method="SLSQP",
                   options={"maxiter": 1000, "ftol": 1e-12})
    return np.clip(res.x, 0, None)


def stats(w):
    r, v = w @ mu, np.sqrt(w @ cov @ w)
    return r, v, (r - RF_ANNUAL) / v if v > 1e-9 else np.nan


def frontier(w_tan, max_short):
    r_tan, v_tan, _ = stats(w_tan)
    line = [(a * (r_tan - RF_ANNUAL) + RF_ANNUAL, a * v_tan) for a in np.linspace(0, 1, 20)]
    r_hi = r_tan + (max_return(max_short) - r_tan) * 0.98
    curve = [stats(solve(target=t, max_short=max_short))[:2] for t in np.linspace(r_tan, r_hi, N_FRONTIER)[1:]]
    return np.array(line + curve).T


w_tan = solve(max_sharpe=True)
w_tan_lo = solve(max_sharpe=True, max_short=0.0)
r_tan, v_tan, s_tan = stats(w_tan)
r_lo, v_lo, s_lo = stats(w_tan_lo)
fc_ret, fc_vol = frontier(w_tan, MAX_SHORT)
fl_ret, fl_vol = frontier(w_tan_lo, 0.0)

print(f"최대샤프(롱·숏):  기대수익 {r_tan:.1%} / 변동성 {v_tan:.1%} / 샤프 {s_tan:.2f}  "
      f"(롱 {w_tan[is_long].sum():.0%}, 숏 {w_tan[is_short].sum():.0%}, 현금 {w_tan[is_cash].sum():.0%})")
print(f"최대샤프(롱온리): 기대수익 {r_lo:.1%} / 변동성 {v_lo:.1%} / 샤프 {s_lo:.2f}")

## 8. 효율적 투자선 그래프

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8.5))
for m in ["kospi", "nasdaq", "crypto"]:
    idx = ((mkt == m).values & is_long)
    ax.scatter(vols[idx], mu[idx], s=14, alpha=0.55, color=MARKET_COLOR[m],
               label=f"{MARKET_LABEL[m]} 개별 자산 롱 ({idx.sum()})")
used_short = is_short & (w_tan > 0.005)
ax.scatter(vols[used_short], mu[used_short], marker="x", s=40, color="crimson",
           label=f"접점에 쓰인 숏 포지션 ({used_short.sum()})")

ax.plot(fl_vol, fl_ret, ":", color="gray", lw=1.8, label="롱온리 투자선 (비교용)")
ax.plot(fc_vol, fc_ret, color="black", lw=2.2,
        label=f"효율적 투자선 (롱·숏, 종목당 숏 ≤ {MAX_SHORT:.0%}, 노출 ≤ {GROSS_LIMIT:g}배)")
ax.scatter(0, RF_ANNUAL, marker="s", s=90, color="gold", edgecolor="black", zorder=6,
           label=f"현금 100%  수익 {RF_ANNUAL:.1%} / 위험 0%")
ax.scatter(v_lo, r_lo, marker="*", s=220, color="gray", zorder=6,
           label=f"최대샤프(롱온리)  수익 {r_lo:.1%} / 위험 {v_lo:.1%} / 샤프 {s_lo:.2f}")
ax.scatter(v_tan, r_tan, marker="*", s=320, color="red", zorder=6,
           label=f"최대샤프(롱·숏)  수익 {r_tan:.1%} / 위험 {v_tan:.1%} / 샤프 {s_tan:.2f}")

for i in np.argsort(-w_tan)[:LABEL_TOP]:
    if w_tan[i] > 0.005 and not is_cash[i]:
        label = pos["name"].iloc[i] + (" 숏" if is_short[i] else "")
        ax.annotate(label, (vols[i], mu[i]), fontsize=8, xytext=(4, 3), textcoords="offset points")

cost_txt = (f"비용 반영: 왕복비용/{HOLDING_YEARS:g}년 차감 (롱 코스피 {ROUND_TRIP['kospi']:.2%}, "
            f"나스닥 {ROUND_TRIP['nasdaq']:.2%}, 코인 {ROUND_TRIP['crypto']:.2%} / 숏 나스닥 "
            f"{SHORT_ROUND_TRIP['nasdaq']:.2%}, 코인 {SHORT_ROUND_TRIP['crypto']:.2%})" if APPLY_COST else "비용 미반영")
ax.set_title(f"효율적 투자선 — 위험자산 {is_long.sum()}종 (숏 가능 {is_short.sum()}종) + 원화 현금, 원화 기준, "
             f"{rets.index[0]:%Y.%m}~{rets.index[-1]:%Y.%m} 주간수익률\n{cost_txt}", fontsize=12)
ax.set_xlabel("연 변동성 (위험)")
ax.set_ylabel("연 기대수익률")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.set_xlim(-0.01, np.percentile(vols[is_long], 97) * 1.1)
ax.set_ylim(min(mu[is_long].min(), 0) - 0.1, max(mu[is_long].max(), fc_ret.max()) * 1.15)
ax.axhline(0, color="lightgray", lw=0.8)
ax.grid(alpha=0.3)
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout()
fig.savefig(OUT_PNG, dpi=150)
plt.show()

## 9. 포트폴리오 비중
접점보다 아래(덜 위험한) 지점은 **현금 + 최대샤프 포트폴리오**의 조합이다.
예: 목표 변동성 σ면 접점 포트폴리오를 σ / σ_접점 배로 축소하고, 나머지는 현금

In [ ]:
weights = pd.DataFrame({
    "name": pos["name"].values,
    "market": mkt.map(MARKET_LABEL).values,
    "side": side,
    "exp_return": mu,
    "volatility": vols,
    "w_max_sharpe": w_tan,
    "w_max_sharpe_long_only": w_tan_lo,
}, index=names).sort_values("w_max_sharpe", ascending=False)
weights.to_csv(OUT_CSV, encoding="utf-8-sig", float_format="%.6f")

def pct(df):
    df = df.copy()
    for c in ["exp_return", "volatility", "w_max_sharpe", "w_max_sharpe_long_only"]:
        df[c] = df[c].map("{:.1%}".format)
    return df

print("[최대샤프 포트폴리오 (롱·숏)] 비중 0.5% 이상")
pct(weights[weights["w_max_sharpe"] > 0.005])

In [ ]:
print("[최대샤프 포트폴리오 (롱온리, 비교용)] 비중 0.5% 이상")
weights[weights["w_max_sharpe_long_only"] > 0.005].sort_values("w_max_sharpe_long_only", ascending=False).pipe(pct)

## 10. 목표 변동성별 현금 + 접점 포트폴리오 배분

In [ ]:
rows = []
# 접점보다 위험한 구간은 노출 한도를 다 쓴 곡선이므로 여기서는 접점까지만 표시
long_tan, short_tan = w_tan[is_long].sum(), w_tan[is_short].sum()
for sigma in [s for s in np.arange(0.01, v_tan, 0.01)] + [v_tan]:
    a = sigma / v_tan
    rows.append({"목표 변동성": f"{sigma:.1%}", "롱 비중": f"{a * long_tan:.0%}", "숏 비중": f"{a * short_tan:.0%}",
                 "현금 비중": f"{1 - a * long_tan:.0%}",
                 "기대수익": f"{RF_ANNUAL + a * (r_tan - RF_ANNUAL):.1%}",
                 "1억 기준 롱": f"{a * long_tan * 1e8:,.0f}원", "1억 기준 숏": f"{a * short_tan * 1e8:,.0f}원"})
pd.DataFrame(rows)

## 주의
- 기대수익률은 과거 3년 평균을 그대로 쓴 **표본 내(in-sample)** 추정치. 해당 기간 반도체·방산 등 강세장이 반영돼 수익률·샤프비율이 크게 과대평가됨
- 숏을 허용하면 최적화가 **과거에 떨어진 종목을 숏**으로 고르므로 추정 오차가 더 커짐. 숏은 손실 한도가 없고(가격 상승에 무한 손실) 급등 위험이 있음
- 숏 대여비용·증거금·강제 환매수 규칙은 문서에 없으므로 `get_rules`로 확인한 뒤 `SHORT_BORROW`·`SHORT_ROUND_TRIP`에 반영해야 함
- 미래 성과를 보장하지 않음. `MAX_WEIGHT`를 0.1~0.2로, `MAX_SHORT`를 0.05 이하로 낮추면 종목 비중 한도(지침 9번)를 반영한 더 분산된 결과를 얻을 수 있음